# 11 — Persistent vector retrieval and reproducible benchmarks

This notebook compares the v0.7 JSON reference with SQLite-backed exact search, then evaluates the item-level service and measures local search latency. It uses synthetic vectors and the original fictional food images. No models are downloaded.

**Scope:** deterministic IDs, float32 storage, manifest hashes, exact-result agreement, RRF integration, and a bounded benchmark. The optional FAISS experiment is documented in the guide and requires a separate installation.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from savoroute.data.catalog import load_catalog
from savoroute.media.ingestion import prepare_image_batch, write_image_batch
from savoroute.retrieval.vector import build_vector_index, search_vectors
from savoroute.retrieval.dense import (
    convert_vector_index, load_dense_index, search_dense, validate_dense_catalog
)
from savoroute.retrieval.fusion import search_fused
from savoroute.retrieval.lexical import SearchFilters
from savoroute.evaluation.dense import benchmark_dense
from savoroute.evaluation.fusion import evaluate_fusion
from savoroute.evaluation.retrieval import load_benchmark

SAMPLE = Path('data/sample')
work = TemporaryDirectory(prefix='savoroute-notebook-')
workspace = Path(work.name)
print('Local scratch:', workspace)

## 1. Prepare two local images

The media pipeline produces a catalog with media IDs and normalized PNGs. The encoder below is a deterministic contract fixture, **not** a pretrained image or text model.

In [ ]:
class DemoEncoder:
    model_id = 'notebook-fixture'
    embedding_space_id = 'notebook-fixture@v1'

    def embed_texts(self, texts):
        return [[0.0, 1.0] if 'mushroom' in value.lower() else [1.0, 0.0]
                for value in texts]

    def embed_images(self, paths):
        return [[0.0, 1.0] if 'p002' in path.name else [1.0, 0.0]
                for path in paths]

original = load_catalog(SAMPLE / 'catalog.json')
media_path = write_image_batch(
    prepare_image_batch(SAMPLE / 'image_manifest.json', original), workspace / 'media'
)
catalog = load_catalog(media_path / 'catalog.json')
reference = build_vector_index(catalog, media_path, DemoEncoder())
print('Records:', len(reference.records), 'dimensions:', reference.dimension)

## 2. Convert the reference index

Vectors are written as float32 in a SQLite file. The bundle's manifest records catalog and storage hashes. It cannot be overwritten in place.

In [ ]:
bundle = convert_vector_index(reference, workspace / 'sqlite-bundle')
restored = load_dense_index(bundle.root)
validate_dense_catalog(restored, catalog, catalog_dir=media_path)
print('Engine:', restored.engine, 'records:', restored.count)
print('Bundle files:', sorted(path.name for path in restored.root.iterdir()))

## 3. Compare exact results

The JSON baseline uses Python precision; the SQLite store holds float32. Small numeric differences are expected. Stable record IDs should agree for this fixture.

In [ ]:
query_vector = [0.0, 1.0]
json_hits = search_vectors(reference, query_vector, target='image', k=2)
sql_hits = search_dense(bundle, query_vector, target='image', k=2)
assert [hit.record_id for hit in json_hits] == [hit.record_id for hit in sql_hits]
[(a.record_id, round(a.score, 6), round(b.score, 6))
 for a, b in zip(json_hits, sql_hits, strict=True)]

## 4. Reuse the existing RRF service

The same `search_fused` API ranks canonical items whether the vector source is JSON or SQLite. All filtering occurs at the item level.

In [ ]:
kwargs = dict(encoder=DemoEncoder(), catalog_dir=media_path,
              filters=SearchFilters(kind='recipe'), limit=4)
ranked_reference = search_fused(catalog, 'mushroom pasta', index=reference, **kwargs)
ranked_sqlite = search_fused(catalog, 'mushroom pasta', index=bundle, **kwargs)
assert [h.item.item_id for h in ranked_reference] == [h.item.item_id for h in ranked_sqlite]
[(h.item.item_id, [e.source for e in h.evidence]) for h in ranked_sqlite]

## 5. Evaluate a more varied *fictional* judgment set

The 42-item challenge includes literal, paraphrase, ambiguous multi-answer and negative query categories. These judgments are not independent real-world annotations.

In [ ]:
judgments = load_benchmark(SAMPLE / 'relevance_challenge_v1.json')
result = evaluate_fusion(catalog, judgments, index=bundle,
                         encoder=DemoEncoder(), catalog_dir=media_path, at_k=5)
print({name: value for name, value in result.as_dict().items() if name != 'cases'})

## 6. Local latency and exact agreement

This separate microbenchmark uses randomly generated synthetic unit vectors. It includes backend query overhead but excludes embedding generation. Report the machine and library versions when comparing results.

In [ ]:
report = benchmark_dense(backend='sqlite', records=400, dimension=24,
                         queries=10, k=5, seed=7)
print(report.as_dict())
assert report.mean_exact_overlap_at_k == 1.0

## 7. Reproducibility and limitations

The optional `faiss-hnsw` backend requires `pip install -e ".[ann]"`. Use `savoroute backend benchmark --engine faiss-hnsw ...` on your own machine to report ANN agreement and latency against exact results. FAISS graph retrieval may be approximate, and postfilters can reduce recall.

SQLite exact search still scans vectors linearly; the full application retains Python BM25. See `docs/dense-index.md` and `docs/relevance-annotation.md` before making scale or accuracy claims.

In [ ]:
work.cleanup()
print('Scratch directory removed.')